In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, auc
import os
import pickle

import scanpy as sc 
import json
import re
import pyranges as pr
from cellgrn.utils import enhancer_eval, eval_gene_peak, eval_tf_recovery, eval_tf_recovery_ctx, eval_tf_gene, load_scenic2, load_linger_ctx,load_linger_all,load_thres_grn

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [5]:
suffix = "scale2"

soft_res = {}
soft_res_ctx = {}

soft = f'linger_metacell_{suffix}_samp'
gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_pbmc_metacell_linger/",
                                                    scale="sample",suffix=suffix,peak_rev=True)
soft_res[soft] = {}
soft_res[soft]['grn_res'] = grn_res
soft_res[soft]['gene_peak_res'] = gene_peak_res
soft_res[soft]['tf_peak_res'] = tf_peak_res

soft = f'linger_metacell_{suffix}_ctx'
gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_pbmc_metacell_linger/",
                                                    scale='celltype',suffix=suffix,peak_rev=True)
soft_res_ctx[soft] = {}
soft_res_ctx[soft]['grn_res'] = grn_res
soft_res_ctx[soft]['gene_peak_res'] = gene_peak_res
soft_res_ctx[soft]['tf_peak_res'] = tf_peak_res

soft = f'linger_thres_{suffix}_samp'
gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_pbmc_linger/",
                                                    scale="sample",suffix=suffix,peak_rev=True)
soft_res[soft] = {}
soft_res[soft]['grn_res'] = grn_res
soft_res[soft]['gene_peak_res'] = gene_peak_res
soft_res[soft]['tf_peak_res'] = tf_peak_res

soft = f'linger_thres_{suffix}_ctx'
gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_pbmc_linger/",
                                                    scale='celltype',suffix=suffix,peak_rev=True)
soft_res_ctx[soft] = {}
soft_res_ctx[soft]['grn_res'] = grn_res
soft_res_ctx[soft]['gene_peak_res'] = gene_peak_res
soft_res_ctx[soft]['tf_peak_res'] = tf_peak_res


In [6]:
soft_res_ctx.keys()

dict_keys(['linger_metacell_scale2_ctx', 'linger_thres_scale2_ctx'])

In [4]:
outdir = "/home/shaliu_fu/multireg/cellGRN/eval/results/pbmc_metacell/"
os.system(f"mkdir -p {outdir}")

0

In [ ]:

ps_data = np.load('/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/pseudo_data_pstime.npz',allow_pickle=True)

In [ ]:
scRNA_data = ps_data['rna']
scATAC_data = ps_data['atac']

In [ ]:
input_rna = sc.read_h5ad(f"/home/shaliu_fu/multireg/benchmark/bench_dataset/10X_PBMC/PBMC-multiome-raw-RNA-counts.h5ad")
input_atac = sc.read_h5ad(f"/home/shaliu_fu/multireg/benchmark/bench_dataset/10X_PBMC/PBMC-multiome-raw-ATAC-peaks.h5ad")
input_df1 =input_rna.X.toarray()
input_df2 = input_atac.X.toarray()

In [ ]:
out_res = pd.DataFrame({"n_cell":[input_df1.shape[0],scRNA_data.shape[0]],
                        "scRNA_nzero":[round(1-np.sum(input_df1 == 0)/input_df1.size,3), round(1-np.sum(scRNA_data == 0)/scRNA_data.size,3)],
                        "scATAC_nzero":[round(1-np.sum(input_df2 == 0)/input_df2.size,3), round(1-np.sum(scATAC_data == 0)/scATAC_data.size,3)],
                        })
out_res2 = out_res.T
out_res2.columns = ["Single_cell","Metacell"]
out_res2.to_csv(f"{outdir}/metacell_summary.csv")

,0,1
n_cell,15021.000,1504.000
scRNA_nzero,0.050,0.198
scATAC_nzero,0.054,0.263


In [5]:
cd4_gold = pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/pbmc/10X_PBMC_CD4_STARR.bed",sep='\t',header=None)
cd4_gold['Peak'] = cd4_gold.apply(lambda row:f"{row[0]}:{row[1]}-{row[2]}", axis=1)

In [6]:
pr_curve = pd.DataFrame()
res_summary = []
for soft in soft_res.keys():
    gene_peak_res = soft_res[soft]['gene_peak_res'].copy()

    if gene_peak_res is not None:
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, gene_peak_res,soft)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])



for soft in soft_res_ctx.keys():
    gene_peak_res = soft_res_ctx[soft]['gene_peak_res']
    if gene_peak_res is not None:
        gene_peak_res_ = gene_peak_res[gene_peak_res['cell_type']=='CD4 T']
        soft_ = f"{soft}_CD4"
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, gene_peak_res,soft_)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft_,round(pr_auc,5),round(epr,5),round(f1,5)])


res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]

out_res = res_summary
out_res.to_csv(f"{outdir}/cd4_enhancer_gene_peak_res.csv",index=False,header=True)

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retur

In [7]:

pr_curve = pd.DataFrame()
res_summary = []
for soft in soft_res.keys():
    tf_peak_res = soft_res[soft]['tf_peak_res']

    if tf_peak_res is not None:
        tf_peak_res = tf_peak_res.nlargest(20000,"Score")
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, tf_peak_res,soft)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])



for soft in soft_res_ctx.keys():
    tf_peak_res = soft_res_ctx[soft]['tf_peak_res']
    if tf_peak_res is not None:
        tf_peak_res_ = tf_peak_res[tf_peak_res['cell_type']=='CD4 T']
        tf_peak_res_ = tf_peak_res_.nlargest(20000,"Score")
        soft_ = f"{soft}_CD4"
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, tf_peak_res_,soft_)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft_,round(pr_auc,5),round(epr,5),round(f1,5)])

    
res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]

out_res = res_summary
out_res.to_csv(f"{outdir}/cd4_enhancer_tf_peak_res.csv",index=False,header=True)

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retur

In [8]:
from scipy.sparse import load_npz

gene_peak_link = load_npz('/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/gene_peak_dist_all.npz')
input_gene = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_gene.txt")]
input_peak = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_peak.txt")]
gene_peak_coo = gene_peak_link.tocoo()
peak_idx = gene_peak_coo.col   # [n_nonzero]
gene_idx = gene_peak_coo.row   # [n_nonzero]

gene_peak_dist = pd.DataFrame({"Gene" : pd.Series(input_gene).values[gene_idx],
                              "Peak" : pd.Series(input_peak).values[peak_idx],
                              "Dist": gene_peak_coo.data
})
gene_peak_dist['Peak'] = gene_peak_dist['Peak'].str.replace(r'^([^-\s]+)-', r'\1:', regex=True)

In [ ]:

ctx_dict = {"CD4 T":"cd4","B":"b","CD8 T":"cd8","NK":"nk"}


res_summary = []
range_summary = pd.DataFrame()
pr_curve = pd.DataFrame()

for ctx in ctx_dict.keys():
    hic_gold = f"/home/shaliu_fu/multireg/benchmark/datasets/pbmc/encode_hic/{ctx_dict[ctx]}_gene_peak_gold.bed"
    gold_pr_region = pr.read_bed(hic_gold)  
    for soft in soft_res.keys():

        gene_peak_res = soft_res[soft]['gene_peak_res']
        # soft_ = f"{soft}_{ctx}"
        if gene_peak_res is not None:
            gene_peak_res_ = pd.merge(gene_peak_res,gene_peak_dist,on=["Gene","Peak"],how="left")


            pr_table,pr_auc,epr,f1, range_res = eval_gene_peak(gold_pr_region, gene_peak_res_,gene_peak_dist, soft)
            pr_table['celltype']=ctx
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])
            # pr_summary.append([soft,round(pr_auc,5),ctx])
            # epr_summary.append([soft,round(epr,5),ctx])
            range_res['celltype']=ctx
            range_summary = pd.concat([range_summary,range_res], axis=0)

    for soft in soft_res_ctx.keys():
        gene_peak_res = soft_res_ctx[soft]['gene_peak_res']
        if gene_peak_res is not None:
            gene_peak_res_ = gene_peak_res[gene_peak_res['cell_type']==ctx]
            gene_peak_res_ = pd.merge(gene_peak_res_,gene_peak_dist,on=["Gene","Peak"],how="left")
            # soft_ = f"{soft}_{ctx}"
            # pr_table,pr_auc,epr = enhancer_eval(cd4_gold, gene_peak_res_,soft_)
            pr_table,pr_auc,epr,f1, range_res = eval_gene_peak(gold_pr_region, gene_peak_res_,gene_peak_dist, soft)
            pr_table['celltype']=ctx
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            range_res['celltype']=ctx

            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])

            range_summary = pd.concat([range_summary,range_res], axis=0)

res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1","celltype"]

out_res = res_summary
out_res.to_csv(f"{outdir}/pbmc_ctx_hic_res.csv",index=False,header=True)

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

In [10]:

ctx_dict = {"CD4 T":"CD4T","B":"B","CD8 T":"CD8T","NK":"NK","Mono":"monocyte"}

# ctx = "CD4 T"
# hic_gold = f"/home/shaliu_fu/multireg/benchmark/datasets/pbmc/sc-eQTLGen/{ctx_dict[ctx]}_eQTL_gold.txt"

res_summary = []
range_summary = pd.DataFrame()
pr_curve = pd.DataFrame()

for ctx in ctx_dict.keys():
# for ctx in ['CD4 T']:
    hic_gold = f"/home/shaliu_fu/multireg/benchmark/datasets/pbmc/sc-eQTLGen/{ctx_dict[ctx]}_eQTL_gold.txt"

    gold_pr_region = pr.read_bed(hic_gold)
    sel_gene = gold_pr_region.df['Name']

    for soft in soft_res.keys():
    # for soft in ['FigR']:
        gene_peak_res = soft_res[soft]['gene_peak_res']
    # for soft in spa_res.keys():
    #     gene_peak_res = spa_res[soft]

        if gene_peak_res is not None:
            gene_peak_res_ = pd.merge(gene_peak_res,gene_peak_dist,on=["Gene","Peak"],how="left")
            gene_peak_res_ = gene_peak_res_[gene_peak_res_['Gene'].isin(sel_gene)]
            print(f"{soft}: {gene_peak_res_.shape}")
            pr_table,pr_auc,epr,f1, range_res = eval_gene_peak(gold_pr_region, gene_peak_res_,gene_peak_dist, soft)
            pr_table['celltype'] = ctx
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)

            range_res['celltype'] = ctx
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])
            range_summary = pd.concat([range_summary,range_res], axis=0)

    for soft in soft_res_ctx.keys():
        gene_peak_res = soft_res_ctx[soft]['gene_peak_res'] 
        if gene_peak_res is not None:
            # if "cell_type" in gene_peak_res.columns.values:
            gene_peak_res_ = gene_peak_res[gene_peak_res['cell_type']==ctx]
            gene_peak_res_ = pd.merge(gene_peak_res_,gene_peak_dist,on=["Gene","Peak"],how="left")

            pr_table,pr_auc,epr,f1, range_res = eval_gene_peak(gold_pr_region, gene_peak_res_,gene_peak_dist, soft)

            pr_table['celltype'] = ctx
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            range_res['celltype'] = ctx
            
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])
            range_summary = pd.concat([range_summary,range_res], axis=0)


res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1","celltype"]
res_summary.to_csv(f"{outdir}/pbmc_ctx_eQTL_res.csv",index=False,header=True)

linger_metacell_scale2_samp: (946, 4)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

linger_thres_scale2_samp: (939, 4)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

linger_metacell_scale2_samp: (937, 4)


...

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

linger_metacell_scale2_samp: (947, 4)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

linger_thres_scale2_samp: (940, 4)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:300: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:303: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:307: FutureWarning: Series.__getitem__ treating keys a

In [ ]:
with open("/home/shaliu_fu/multireg/multigrn/input_data/gold_dataset/pbmc_gold.pkl", "rb") as f:
    gold_data = pickle.load(f)


tf_gold = {
    "B": [
        "PAX5",    
        "EBF1",    
        "POU2F2",  
        "BCL6",    
        "IRF4"     
    ],
    "CD4 T": [
        "TCF7",   
        "LEF1",    
        "TBX21",   
        "GATA3",   
        "FOXP3",  
        "RORC"     
    ],
    "CD8 T": [
        "RUNX3",  
        "EOMES",  
        "TBX21",  
        "PRDM1"   
    ],
    "DC": [
        "TCF4",   
        "IRF8",   
        "BATF3",  
        "IRF4",  
        "ZEB2"   
    ],
    "Mono": [
        "SPI1",    
        "CEBPB",   
        "MAFB",   
        "KLF4",   
        "IRF8"    
    ],
    "NK": [
        "EOMES",   
        "TBX21", 
        "ID2",    
        "IKZF3"   
    ]
}
tf_knock_gold = gold_data['tf_gene_gold']
tf_baseline = gold_data['tf_gene_baseline']

In [12]:


rec_summary = pd.DataFrame()
_, tf_rec_base = eval_tf_recovery(grn_res=tf_baseline,gold_data=tf_gold,label="Pearson",log=False) # ctx dataframe
rec_summary = pd.concat([rec_summary,tf_rec_base],axis=0)


for soft in soft_res.keys():

    tf_gene_res = soft_res[soft]['grn_res']
    tf_gene_res = tf_gene_res[["TF","Gene","Score"]]
    tf_gene_res = tf_gene_res.groupby(['TF', 'Gene'])['Score'].max().reset_index() #只保留最高的。
    if tf_gene_res is not None:
        
        tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")
        _, tf_rec_res = eval_tf_recovery(grn_res=tf_gene_res2,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)


for soft in soft_res_ctx.keys():

    tf_gene_res = soft_res_ctx[soft]['grn_res']
    if tf_gene_res is not None:        
        _, tf_rec_res = eval_tf_recovery_ctx(grn_res_ctx=tf_gene_res,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)

rec_summary.to_csv(f"{outdir}/tf_marker_tf_gene_res.csv",index=True,header=True)

In [ ]:

rec_summary = pd.DataFrame()


for soft in soft_res.keys():

    tf_peak_res = soft_res[soft]['tf_peak_res']
    if tf_peak_res is not None:
        tf_peak_res = tf_peak_res[["TF","Peak","Score"]]
        tf_peak_res = tf_peak_res.groupby(['TF', 'Peak'])['Score'].max().reset_index() #只保留最高的。
        
        tf_peak_res2 = tf_peak_res.nlargest(20000,"Score")
        _, tf_rec_res = eval_tf_recovery(grn_res=tf_peak_res2,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)

for soft in soft_res_ctx.keys():

    tf_peak_res = soft_res_ctx[soft]['tf_peak_res']
    
    if tf_peak_res is not None: 
        _, tf_rec_res = eval_tf_recovery_ctx(grn_res_ctx=tf_peak_res,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)

rec_summary.to_csv(f"{outdir}/tf_marker_tf_peak_res.csv",index=True,header=True)

In [14]:
tmp = tf_knock_gold['PBMC_TF_knock']
sel_tf = set([i.split("_")[0] for i in tmp])
sel_tf

tf_knock_gold2 = {}
for tfs in sel_tf:
    tf_knock_gold2[tfs] = []

for i in tf_knock_gold['PBMC_TF_knock']:
    tf = i.split("_")[0]
    gene = i.split("_")[1]
    tf_knock_gold2[tf].append(gene)

In [ ]:


pr_curve = pd.DataFrame()
res_summary = []


for t_gold in tf_knock_gold.keys():
    tf_knock = tf_knock_gold[t_gold]
    tf_baseline2 = tf_baseline[tf_baseline["TF"].isin(sel_tf)]
    pr_auc,epr,f1, pr_table = eval_tf_gene(grn_res=tf_baseline2.nlargest(10000,"Score"),gold_data=tf_knock,
                    all_comb=tf_baseline2.shape[0],label="Pearson" ) # 
    pr_table['celltype'] = ctx
    pr_curve = pd.concat([pr_curve,pr_table],axis=0)
    res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])

    for soft in soft_res.keys():

        tf_gene_res = soft_res[soft]['grn_res']
        if tf_gene_res is not None:
            tf_gene_res = tf_gene_res[["TF","Gene","Score"]]
            tf_gene_res = tf_gene_res.groupby(['TF', 'Gene'])['Score'].max().reset_index() 
            
            # tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")
            tf_gene_res2 = tf_gene_res[tf_gene_res['TF'].isin(sel_tf)]
            tf_gene_res2 =  tf_gene_res2.nlargest(10000,"Score")
            print(f"{t_gold}: {soft}- candidates : {tf_gene_res2.shape[0]}")
            if tf_gene_res2.shape[0] > 1:
                pr_auc,epr,f1, pr_table = eval_tf_gene(grn_res=tf_gene_res2,gold_data=tf_knock,
                            all_comb=tf_baseline2.shape[0],label=soft ) # 
                pr_table['celltype'] = ctx
            else:
                pr_auc = 0
                epr = 0
                f1 = 0
                pr_table = None
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])

res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]

res_summary.to_csv(f"{outdir}/pbmc_tf_knock_res.csv",index=False,header=True)


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)


PBMC_TF_knock: linger_metacell_scale2_samp- candidates : 391
PBMC_TF_knock: linger_thres_scale2_samp- candidates : 421


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
